# SunnyBest Telecommunications — Stockout Analysis
**Analyst:** Bonaventure (Team Lead Overview)

This notebook covers all three focus areas so the team lead can understand the full picture and guide the team:

| Section | Focus | Mirrors |
|---|---|---|
| 1 | Store-Level Analysis | Emma (Data Analyst) |
| 2 | Product-Level Analysis | Chi Chi (Associate Data Scientist) |
| 3 | Timing & Drivers | Udy (AI Developer) |

## 0.Imports

In [1]:
# Install required packages if not already present
import subprocess, sys

packages = [
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'sqlalchemy',
    'psycopg2-binary',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All packages ready.')

All packages ready.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_palette("Set2")

DATA_DIR = "../data/raw"
BRAND_COLOR = "#2196F3"
ALERT_COLOR = "#E53935"

In [ ]:
sales = pd.read_csv(f"{DATA_DIR}/sunnybest_sales.csv", parse_dates=["date"], low_memory=False)
stores = pd.read_csv(f"{DATA_DIR}/sunnybest_stores.csv")
products = pd.read_csv(f"{DATA_DIR}/sunnybest_products.csv")

# merge store name and product name into sales
sales = sales.merge(stores[["store_id", "store_name", "region"]], on="store_id", how="left")
sales = sales.merge(products[["product_id", "product_name", "brand"]], on="product_id", how="left")

# calendar features
sales["year"]        = sales["date"].dt.year
sales["month"]       = sales["date"].dt.month
sales["month_name"]  = sales["date"].dt.strftime("%b")
sales["day_of_week"] = sales["date"].dt.day_name()
sales["week"]        = sales["date"].dt.isocalendar().week.astype(int)
sales["year_month"]  = sales["date"].dt.to_period("M").astype(str)

print(f"Dataset: {len(sales):,} rows | {sales['date'].min().date()} → {sales['date'].max().date()}")
print(f"Stores: {sales['store_id'].nunique()} | Products: {sales['product_id'].nunique()}")
print(f"Total stockout events: {sales['stockout_occurred'].sum():,} ({sales['stockout_occurred'].mean()*100:.1f}% of all records)")

---
## Part 1 — Store-Level Analysis
*Mirrors Emma's focus: Which stores experience the most stockouts? How frequently? Are there trends over time?*

In [ ]:
# Stockout count and rate per store
store_summary = (
    sales.groupby(["store_id", "store_name", "store_size", "region"])
    .agg(
        total_records=("stockout_occurred", "count"),
        stockout_count=("stockout_occurred", "sum"),
    )
    .reset_index()
)
store_summary["stockout_rate"] = store_summary["stockout_count"] / store_summary["total_records"]
store_summary = store_summary.sort_values("stockout_count", ascending=False)

store_summary[["store_name", "store_size", "region", "total_records", "stockout_count", "stockout_rate"]].style.format(
    {"stockout_rate": "{:.1%}", "total_records": "{:,}", "stockout_count": "{:,}"}
).background_gradient(subset=["stockout_rate"], cmap="YlOrRd")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: total stockout count
axes[0].barh(
    store_summary["store_name"],
    store_summary["stockout_count"],
    color=[ALERT_COLOR if s == store_summary["stockout_count"].max() else BRAND_COLOR
           for s in store_summary["stockout_count"]]
)
axes[0].set_xlabel("Stockout Events")
axes[0].set_title("Total Stockout Events by Store", fontweight="bold")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# right: stockout rate
rate_sorted = store_summary.sort_values("stockout_rate", ascending=False)
axes[1].barh(
    rate_sorted["store_name"],
    rate_sorted["stockout_rate"] * 100,
    color=[ALERT_COLOR if r == rate_sorted["stockout_rate"].max() else BRAND_COLOR
           for r in rate_sorted["stockout_rate"]]
)
axes[1].set_xlabel("Stockout Rate (%)")
axes[1].set_title("Stockout Rate by Store", fontweight="bold")

plt.tight_layout()
plt.suptitle("Store-Level Stockout Overview", fontsize=14, fontweight="bold", y=1.02)
plt.show()

In [ ]:
# Monthly stockout trend per store
monthly_store = (
    sales.groupby(["year_month", "store_name"])["stockout_occurred"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"stockout_occurred": "stockout_rate_pct"})
)

plt.figure(figsize=(16, 5))
for store, grp in monthly_store.groupby("store_name"):
    plt.plot(grp["year_month"], grp["stockout_rate_pct"], marker="o", markersize=3, label=store)

plt.xticks(rotation=45, ha="right", fontsize=8)
plt.ylabel("Stockout Rate (%)")
plt.title("Monthly Stockout Rate Trend by Store", fontweight="bold")
plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: store × month stockout rate
heat_data = (
    sales.groupby(["store_name", "month"])["stockout_occurred"]
    .mean()
    .mul(100)
    .unstack("month")
)
heat_data.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                      "Jul","Aug","Sep","Oct","Nov","Dec"]

plt.figure(figsize=(14, 4))
sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label": "Stockout Rate (%)"})
plt.title("Stockout Rate (%) — Store × Month", fontweight="bold")
plt.ylabel("")
plt.tight_layout()
plt.show()

### Store-Level Key Observations

> Fill in after running the charts. Guide questions for Emma:
> - Which store has the highest raw stockout count? Is this because it's the largest store?
> - Which store has the highest **rate** — that's the real problem store.
> - Are there months where stockout rates spike? Does this happen across all stores or just some?
> - Does store size or region explain the pattern?

---
## Part 2 — Product-Level Analysis
*Mirrors Chi Chi's focus: Which products stock out most? High-demand or high-value? Category and promotion patterns?*

In [ ]:
# Stockout rate by category
cat_summary = (
    sales.groupby("category")
    .agg(
        total=("stockout_occurred", "count"),
        stockouts=("stockout_occurred", "sum"),
        avg_units_sold=("units_sold", "mean"),
        avg_price=("regular_price", "mean"),
    )
    .reset_index()
)
cat_summary["stockout_rate"] = cat_summary["stockouts"] / cat_summary["total"]
cat_summary = cat_summary.sort_values("stockout_rate", ascending=False)

cat_summary.style.format({
    "stockout_rate": "{:.1%}",
    "avg_units_sold": "{:.1f}",
    "avg_price": "₦{:,.0f}",
    "total": "{:,}",
    "stockouts": "{:,}",
}).background_gradient(subset=["stockout_rate"], cmap="YlOrRd")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# left: stockout rate by category
axes[0].barh(cat_summary["category"], cat_summary["stockout_rate"] * 100, color=ALERT_COLOR)
axes[0].set_xlabel("Stockout Rate (%)")
axes[0].set_title("Stockout Rate by Category", fontweight="bold")

# right: avg price vs stockout rate (bubble = stockout volume)
axes[1].scatter(
    cat_summary["avg_price"] / 1000,
    cat_summary["stockout_rate"] * 100,
    s=cat_summary["stockouts"] / cat_summary["stockouts"].max() * 800 + 50,
    c=cat_summary["stockout_rate"], cmap="YlOrRd",
    alpha=0.8, edgecolors="grey", linewidth=0.5
)
for _, row in cat_summary.iterrows():
    axes[1].annotate(row["category"].split()[0],
                     (row["avg_price"]/1000, row["stockout_rate"]*100),
                     fontsize=8, ha="center", va="bottom")
axes[1].set_xlabel("Avg Regular Price (₦ thousands)")
axes[1].set_ylabel("Stockout Rate (%)")
axes[1].set_title("Price vs Stockout Rate\n(bubble size = stockout volume)", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 products by stockout count
top_products = (
    sales.groupby(["product_id", "product_name", "category", "brand", "regular_price"])
    .agg(
        stockout_count=("stockout_occurred", "sum"),
        total=("stockout_occurred", "count"),
        avg_units_sold=("units_sold", "mean"),
    )
    .reset_index()
)
top_products["stockout_rate"] = top_products["stockout_count"] / top_products["total"]
top20 = top_products.nlargest(20, "stockout_count")

plt.figure(figsize=(12, 7))
plt.barh(
    top20["product_name"],
    top20["stockout_count"],
    color=[ALERT_COLOR if r > 0.10 else BRAND_COLOR for r in top20["stockout_rate"]]
)
plt.xlabel("Stockout Events")
plt.title("Top 20 Products by Stockout Count\n(red = stockout rate > 10%)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Promotions and stockouts by category
promo_cat = (
    sales.groupby(["category", "promo_flag"])["stockout_occurred"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"stockout_occurred": "stockout_rate_pct", "promo_flag": "promotion"})
)
promo_cat["promotion"] = promo_cat["promotion"].map({0: "No Promo", 1: "On Promo"})

pivot = promo_cat.pivot(index="category", columns="promotion", values="stockout_rate_pct")
pivot["uplift_pp"] = pivot["On Promo"] - pivot["No Promo"]
pivot = pivot.sort_values("uplift_pp", ascending=False)

ax = pivot[["No Promo", "On Promo"]].plot(
    kind="bar", figsize=(12, 5),
    color=[BRAND_COLOR, ALERT_COLOR], edgecolor="white"
)
ax.set_ylabel("Stockout Rate (%)")
ax.set_title("Stockout Rate: Promo vs No-Promo by Category", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.legend(title="")

for i, (_, row) in enumerate(pivot.iterrows()):
    sign = "+" if row["uplift_pp"] >= 0 else ""
    ax.annotate(f"{sign}{row['uplift_pp']:.1f}pp",
                xy=(i, max(row["No Promo"], row["On Promo"]) + 0.2),
                ha="center", fontsize=8, color="black")
plt.tight_layout()
plt.show()

### Product-Level Key Observations

> Fill in after running. Guide questions for Chi Chi:
> - Does a specific category dominate stockouts, or is it spread evenly?
> - Are high-price products (Mobile Phones, Laptops) more prone to stockouts than low-price ones (Accessories, Telecom)?
> - Which categories show the biggest stockout increase when a promotion is running? This signals demand spikes that inventory can't absorb.
> - Are the top 20 stockout products concentrated in one category?

---
## Part 3 — Timing & Drivers
*Mirrors Udy's focus: When do stockouts occur? Do promotions drive them? Is low inventory a consistent driver?*

In [ ]:
# Stockout rate by month and day-of-week
month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

monthly = (
    sales.groupby("month_name")["stockout_occurred"]
    .mean().mul(100)
    .reindex(month_order)
    .reset_index()
    .rename(columns={"stockout_occurred": "rate"})
)

dow = (
    sales.groupby("day_of_week")["stockout_occurred"]
    .mean().mul(100)
    .reindex(day_order)
    .reset_index()
    .rename(columns={"stockout_occurred": "rate"})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(monthly["month_name"], monthly["rate"],
            color=[ALERT_COLOR if r == monthly["rate"].max() else BRAND_COLOR for r in monthly["rate"]])
axes[0].set_ylabel("Stockout Rate (%)")
axes[0].set_title("Stockout Rate by Month", fontweight="bold")

axes[1].bar(dow["day_of_week"], dow["rate"],
            color=[ALERT_COLOR if r == dow["rate"].max() else BRAND_COLOR for r in dow["rate"]])
axes[1].set_ylabel("Stockout Rate (%)")
axes[1].set_title("Stockout Rate by Day of Week", fontweight="bold")
axes[1].set_xticklabels(day_order, rotation=30, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# Stockout rate by season
season_stats = (
    sales.groupby("season")["stockout_occurred"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "stockout_rate", "sum": "stockouts", "count": "total"})
    .reset_index()
    .sort_values("stockout_rate", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [ALERT_COLOR, BRAND_COLOR, "#4CAF50"]
bars = ax.bar(season_stats["season"], season_stats["stockout_rate"] * 100,
              color=colors[:len(season_stats)])
for bar, rate in zip(bars, season_stats["stockout_rate"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{rate:.1%}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("Stockout Rate (%)")
ax.set_title("Stockout Rate by Season", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Promotion effect: overall + monthly correlation
promo_effect = (
    sales.groupby("promo_flag")["stockout_occurred"]
    .agg(["mean", "count"])
    .reset_index()
)
promo_effect["label"] = promo_effect["promo_flag"].map({0: "No Promotion", 1: "Promotion Active"})
promo_effect["stockout_rate_pct"] = promo_effect["mean"] * 100

monthly_promo = (
    sales.groupby("year_month")
    .agg(
        stockout_rate=("stockout_occurred", "mean"),
        promo_days_pct=("promo_flag", "mean"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: overall promo effect
axes[0].bar(promo_effect["label"], promo_effect["stockout_rate_pct"],
            color=[BRAND_COLOR, ALERT_COLOR], width=0.4)
for i, row in promo_effect.iterrows():
    axes[0].text(i, row["stockout_rate_pct"] + 0.05,
                 f"{row['stockout_rate_pct']:.2f}%",
                 ha="center", fontweight="bold")
axes[0].set_ylabel("Stockout Rate (%)")
axes[0].set_title("Stockout Rate: Promo vs No Promo", fontweight="bold")

# right: monthly stockout vs promo coverage
ax2 = axes[1].twinx()
axes[1].plot(monthly_promo["year_month"], monthly_promo["stockout_rate"] * 100,
             color=ALERT_COLOR, label="Stockout Rate (%)", linewidth=2)
ax2.bar(monthly_promo["year_month"], monthly_promo["promo_days_pct"] * 100,
        alpha=0.3, color=BRAND_COLOR, label="Promo Coverage (%)")
axes[1].set_ylabel("Stockout Rate (%)", color=ALERT_COLOR)
ax2.set_ylabel("Promo Coverage (%)", color=BRAND_COLOR)
axes[1].set_title("Monthly Stockout Rate vs Promo Coverage", fontweight="bold")
axes[1].tick_params(axis="x", rotation=45, labelsize=7)
axes[1].legend(loc="upper left", fontsize=8)
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Starting inventory at time of stockout vs no stockout
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stockout_yes = sales[sales["stockout_occurred"] == 1]["starting_inventory"]
stockout_no  = sales[sales["stockout_occurred"] == 0]["starting_inventory"]

axes[0].hist(stockout_no.clip(0, 50), bins=30, alpha=0.6, color=BRAND_COLOR,
             label=f"No Stockout (n={len(stockout_no):,})", density=True)
axes[0].hist(stockout_yes.clip(0, 50), bins=30, alpha=0.6, color=ALERT_COLOR,
             label=f"Stockout (n={len(stockout_yes):,})", density=True)
axes[0].set_xlabel("Starting Inventory (capped at 50)")
axes[0].set_ylabel("Density")
axes[0].set_title("Starting Inventory Distribution\nStockout vs No Stockout", fontweight="bold")
axes[0].legend()

sales["inv_bucket"] = pd.cut(
    sales["starting_inventory"],
    bins=[0, 1, 3, 5, 10, 20, 50, float("inf")],
    labels=["0", "1-3", "3-5", "5-10", "10-20", "20-50", "50+"],
    right=True
)
inv_rate = (
    sales.groupby("inv_bucket", observed=True)["stockout_occurred"]
    .mean().mul(100).reset_index()
    .rename(columns={"stockout_occurred": "stockout_rate"})
)

axes[1].bar(inv_rate["inv_bucket"].astype(str), inv_rate["stockout_rate"], color=ALERT_COLOR)
axes[1].set_xlabel("Starting Inventory Level")
axes[1].set_ylabel("Stockout Rate (%)")
axes[1].set_title("Stockout Rate by Starting Inventory Level", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Supply restrictions and stockouts
if "restriction_active" in sales.columns:
    restr_effect = (
        sales.groupby("restriction_active")["stockout_occurred"]
        .agg(["mean", "count"])
        .reset_index()
    )
    restr_effect["label"] = restr_effect["restriction_active"].map(
        {0: "No Restriction", 1: "Restriction Active"}
    )

    restr_type = (
        sales[sales["restriction_active"] == 1]
        .groupby("restriction_type")["stockout_occurred"]
        .mean().mul(100)
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"stockout_occurred": "stockout_rate"})
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].bar(restr_effect["label"], restr_effect["mean"] * 100,
                color=[BRAND_COLOR, ALERT_COLOR], width=0.4)
    for i, row in restr_effect.iterrows():
        axes[0].text(i, row["mean"] * 100 + 0.2,
                     f"{row['mean']:.1%}", ha="center", fontweight="bold")
    axes[0].set_ylabel("Stockout Rate (%)")
    axes[0].set_title("Stockout Rate: Restriction Active vs None", fontweight="bold")

    axes[1].barh(restr_type["restriction_type"], restr_type["stockout_rate"], color=ALERT_COLOR)
    axes[1].set_xlabel("Stockout Rate (%)")
    axes[1].set_title("Stockout Rate by Restriction Type", fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("restriction_active column not present in this dataset.")

In [ ]:
# Year-over-year stockout trend
yearly = (
    sales.groupby("year")["stockout_occurred"]
    .agg(["mean", "sum"])
    .reset_index()
    .rename(columns={"mean": "stockout_rate", "sum": "stockout_count"})
)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.bar(yearly["year"].astype(str), yearly["stockout_count"],
        color=BRAND_COLOR, alpha=0.6, label="Stockout Count")
ax2.plot(yearly["year"].astype(str), yearly["stockout_rate"] * 100,
         color=ALERT_COLOR, marker="o", linewidth=2, label="Stockout Rate (%)")
ax1.set_ylabel("Stockout Count", color=BRAND_COLOR)
ax2.set_ylabel("Stockout Rate (%)", color=ALERT_COLOR)
ax1.set_title("Year-over-Year Stockout Trend", fontweight="bold")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()

### Timing & Drivers Key Observations

> Fill in after running. Guide questions for Udy:
> - Which month(s) have the highest stockout rate? Is it driven by seasonal demand or promotions?
> - Does the day of week matter? Are weekends worse (higher foot traffic)?
> - Is there a clear positive relationship between promotion activity and stockout rate?
> - What starting inventory level is the cut-off — below which stockout risk shoots up? That number becomes the reorder threshold.
> - Which restriction type (Supply Delay, Stock Restriction, etc.) correlates most with stockouts?

---
## Summary — Key Metrics at a Glance

In [ ]:
overall_rate  = sales["stockout_occurred"].mean()
worst_store   = store_summary.sort_values("stockout_rate", ascending=False).iloc[0]
worst_cat     = cat_summary.iloc[0]
promo_rate    = sales[sales["promo_flag"]==1]["stockout_occurred"].mean()
nopromo_rate  = sales[sales["promo_flag"]==0]["stockout_occurred"].mean()
low_inv_rate  = sales[sales["inv_bucket"]=="0"]["stockout_occurred"].mean()

summary = pd.DataFrame([
    {"Metric": "Overall stockout rate",               "Value": f"{overall_rate:.1%}"},
    {"Metric": "Worst store (by rate)",               "Value": f"{worst_store['store_name']} — {worst_store['stockout_rate']:.1%}"},
    {"Metric": "Highest-risk category",               "Value": f"{worst_cat['category']} — {worst_cat['stockout_rate']:.1%}"},
    {"Metric": "Stockout rate on promo days",         "Value": f"{promo_rate:.1%}"},
    {"Metric": "Stockout rate on non-promo days",     "Value": f"{nopromo_rate:.1%}"},
    {"Metric": "Promo uplift (percentage points)",    "Value": f"+{(promo_rate - nopromo_rate)*100:.2f} pp"},
    {"Metric": "Stockout rate when starting inv = 0", "Value": f"{low_inv_rate:.1%}"},
])

summary.style.set_properties(**{"text-align": "left"}).hide(axis="index")